# Fractional Factorial Analysis for Neuro-Symbolic Pipeline

This notebook analyzes results from a $2^{4-1}_{IV}$ fractional factorial experiment evaluating a neuro-symbolic reasoning pipeline.

## Experimental Design

**Factors:**
- **T1 (`use_openie`)**: Use Stanford CoreNLP OpenIE relation triples during text-to-logic conversion
- **T2 (`use_enrichment_kb`)**: Apply enrichment (modal-pair verification, negation detection, finite-domain auxiliaries, conflict resolution)
- **Q1 (`use_shortcuts`)**: Activate deterministic shortcut detectors (modal opposites, lexical antonyms, implication contradictions)
- **Q2 (`expand_query`)**: Create query alternatives using WordNet synonyms + majority-vote ensembling

**Generator**: $T_2 = Q_1 \cdot Q_2 \cdot T_1$

**Datasets**: LogiQA2, LogicBench, DocNLI, Alice

**Response**: Accuracy (4-way classification: entailment, contradiction, uncertain, not_mentioned)

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy scipy statsmodels matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

## 1. Define the Fractional Factorial Design

The $2^{4-1}_{IV}$ design with generator $T_2 = Q_1 \cdot Q_2 \cdot T_1$ has resolution IV, meaning:
- All main effects are estimable and unconfounded with each other
- Main effects are confounded only with 3-factor interactions
- Two-factor interactions are confounded with other two-factor interactions

**Aliasing structure** (defining relation $I = Q_1 Q_2 T_1 T_2$):
- $Q_1 \cdot Q_2$ is aliased with $T_1 \cdot T_2$
- $Q_1 \cdot T_1$ is aliased with $Q_2 \cdot T_2$
- $Q_1 \cdot T_2$ is aliased with $Q_2 \cdot T_1$

In [ ]:
# Define the 2^{4-1} fractional factorial design
# Generator: T2 = Q1 * Q2 * T1
# Using -1 for low level, +1 for high level

design = pd.DataFrame({
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'Q1_shortcuts': [-1, +1, -1, +1, -1, +1, -1, +1],
    'Q2_expand': [-1, -1, +1, +1, -1, -1, +1, +1],
    'T1_openie': [-1, -1, -1, -1, +1, +1, +1, +1],
})

# T2 = Q1 * Q2 * T1 (generator)
design['T2_enrich'] = design['Q1_shortcuts'] * design['Q2_expand'] * design['T1_openie']

# Verify the design matches the paper's Table
print("Fractional Factorial Design (2^{4-1}_IV):")
print("="*60)
design_display = design.copy()
design_display.columns = ['Run', 'Q1:Shortcuts', 'Q2:Expand', 'T1:OpenIE', 'T2:Enrich']
# Convert -1/+1 to -/+ for display
for col in ['Q1:Shortcuts', 'Q2:Expand', 'T1:OpenIE', 'T2:Enrich']:
    design_display[col] = design_display[col].map({-1: '-', 1: '+'})
print(design_display.to_string(index=False))

In [ ]:
# Verify aliasing structure
print("\nAliasing Structure Verification:")
print("="*60)
print(f"Defining relation I = Q1*Q2*T1*T2: {(design['Q1_shortcuts'] * design['Q2_expand'] * design['T1_openie'] * design['T2_enrich']).unique()}")
print("\nAliased pairs (confounded 2-factor interactions):")
print(f"  Q1*Q2 aliased with T1*T2: {np.allclose(design['Q1_shortcuts']*design['Q2_expand'], design['T1_openie']*design['T2_enrich'])}")
print(f"  Q1*T1 aliased with Q2*T2: {np.allclose(design['Q1_shortcuts']*design['T1_openie'], design['Q2_expand']*design['T2_enrich'])}")
print(f"  Q1*T2 aliased with Q2*T1: {np.allclose(design['Q1_shortcuts']*design['T2_enrich'], design['Q2_expand']*design['T1_openie'])}")

## 2. Input Your Experimental Results

**Instructions**: Replace the placeholder values below with your actual accuracy results.

Each cell should contain the accuracy (as a proportion 0-1 or percentage 0-100) for that run × dataset combination.

In [ ]:
# ============================================================================
# INPUT YOUR RESULTS HERE
# ============================================================================
# Enter accuracy values (0-100 scale or 0-1 scale, the code will handle both)
# Rows correspond to Runs 1-8, columns to datasets

results_data = {
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    
    # LogiQA2 accuracies for each run
    'LogiQA2': [
        0.0,  # Run 1: Q1-, Q2-, T1-, T2-
        0.0,  # Run 2: Q1+, Q2-, T1-, T2+
        0.0,  # Run 3: Q1-, Q2+, T1-, T2+
        0.0,  # Run 4: Q1+, Q2+, T1-, T2-
        0.0,  # Run 5: Q1-, Q2-, T1+, T2+
        0.0,  # Run 6: Q1+, Q2-, T1+, T2-
        0.0,  # Run 7: Q1-, Q2+, T1+, T2-
        0.0,  # Run 8: Q1+, Q2+, T1+, T2+
    ],
    
    # LogicBench accuracies for each run
    'LogicBench': [
        0.0,  # Run 1
        0.0,  # Run 2
        0.0,  # Run 3
        0.0,  # Run 4
        0.0,  # Run 5
        0.0,  # Run 6
        0.0,  # Run 7
        0.0,  # Run 8
    ],
    
    # DocNLI accuracies for each run
    'DocNLI': [
        0.0,  # Run 1
        0.0,  # Run 2
        0.0,  # Run 3
        0.0,  # Run 4
        0.0,  # Run 5
        0.0,  # Run 6
        0.0,  # Run 7
        0.0,  # Run 8
    ],
    
    # Alice accuracies for each run
    'Alice': [
        0.0,  # Run 1
        0.0,  # Run 2
        0.0,  # Run 3
        0.0,  # Run 4
        0.0,  # Run 5
        0.0,  # Run 6
        0.0,  # Run 7
        0.0,  # Run 8
    ],
}

results = pd.DataFrame(results_data)

# Merge with design matrix
df = design.merge(results, on='Run')

# Convert to 0-1 scale if needed (detect if values > 1)
datasets = ['LogiQA2', 'LogicBench', 'DocNLI', 'Alice']
for ds in datasets:
    if df[ds].max() > 1:
        df[ds] = df[ds] / 100.0

print("Combined Design + Results:")
print(df.to_string(index=False))

## 3. Compute Main Effects and Interactions

In [ ]:
def compute_effects(df, response_col):
    """
    Compute main effects and two-factor interactions for a 2^{4-1} design.
    
    For a 2-level design, the effect of factor A is:
        Effect(A) = mean(Y | A=+1) - mean(Y | A=-1)
    
    For interactions:
        Effect(A*B) = 0.5 * [mean(Y | A=B) - mean(Y | A≠B)]
    """
    effects = {}
    factors = ['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich']
    
    # Main effects
    for factor in factors:
        high = df[df[factor] == 1][response_col].mean()
        low = df[df[factor] == -1][response_col].mean()
        effects[factor] = high - low
    
    # Two-factor interactions (remember aliasing!)
    # Q1*Q2 (aliased with T1*T2)
    df['Q1_Q2'] = df['Q1_shortcuts'] * df['Q2_expand']
    effects['Q1*Q2 (=T1*T2)'] = df[df['Q1_Q2'] == 1][response_col].mean() - df[df['Q1_Q2'] == -1][response_col].mean()
    
    # Q1*T1 (aliased with Q2*T2)
    df['Q1_T1'] = df['Q1_shortcuts'] * df['T1_openie']
    effects['Q1*T1 (=Q2*T2)'] = df[df['Q1_T1'] == 1][response_col].mean() - df[df['Q1_T1'] == -1][response_col].mean()
    
    # Q1*T2 (aliased with Q2*T1)
    df['Q1_T2'] = df['Q1_shortcuts'] * df['T2_enrich']
    effects['Q1*T2 (=Q2*T1)'] = df[df['Q1_T2'] == 1][response_col].mean() - df[df['Q1_T2'] == -1][response_col].mean()
    
    return effects


def display_effects(effects, dataset_name):
    """Display effects in a formatted table."""
    print(f"\n{'='*60}")
    print(f"Effects for {dataset_name}")
    print(f"{'='*60}")
    print(f"{'Effect':<25} {'Estimate':>12} {'Interpretation':>20}")
    print("-"*60)
    for name, value in effects.items():
        interpretation = '+' if value > 0 else '-' if value < 0 else '0'
        magnitude = 'Strong' if abs(value) > 0.1 else 'Moderate' if abs(value) > 0.05 else 'Weak'
        print(f"{name:<25} {value:>+12.4f} {magnitude + ' ' + interpretation:>20}")
    return effects

In [ ]:
# Compute and display effects for each dataset
all_effects = {}
for dataset in datasets:
    effects = compute_effects(df.copy(), dataset)
    display_effects(effects, dataset)
    all_effects[dataset] = effects

In [ ]:
# Compute average effect across all datasets
df['Average'] = df[datasets].mean(axis=1)
avg_effects = compute_effects(df.copy(), 'Average')
display_effects(avg_effects, 'AVERAGE (All Datasets)')

## 4. Statistical Significance Testing

In [ ]:
def run_anova(df, response_col):
    """
    Run ANOVA for the fractional factorial design.
    Note: With no replication, we use interactions as error estimate.
    """
    # Create coded factor columns
    df_anova = df.copy()
    df_anova['Y'] = df_anova[response_col]
    
    # Fit main effects model
    formula = 'Y ~ Q1_shortcuts + Q2_expand + T1_openie + T2_enrich'
    model = ols(formula, data=df_anova).fit()
    
    return model


def compute_effect_significance(df, response_col, n_bootstrap=10000):
    """
    Use bootstrap to estimate confidence intervals for effects.
    With only 8 observations, traditional ANOVA p-values are unreliable.
    """
    np.random.seed(42)
    original_effects = compute_effects(df.copy(), response_col)
    
    bootstrap_effects = {key: [] for key in original_effects.keys()}
    
    for _ in range(n_bootstrap):
        # Resample rows with replacement
        boot_df = df.sample(n=len(df), replace=True)
        boot_effects = compute_effects(boot_df.copy(), response_col)
        for key in bootstrap_effects:
            bootstrap_effects[key].append(boot_effects[key])
    
    # Compute 95% CIs and pseudo p-values
    results = []
    for key in original_effects:
        boot_array = np.array(bootstrap_effects[key])
        ci_low = np.percentile(boot_array, 2.5)
        ci_high = np.percentile(boot_array, 97.5)
        # Pseudo p-value: proportion of bootstrap samples with opposite sign
        if original_effects[key] > 0:
            p_value = np.mean(boot_array <= 0) * 2  # Two-tailed
        else:
            p_value = np.mean(boot_array >= 0) * 2
        p_value = min(p_value, 1.0)
        
        results.append({
            'Effect': key,
            'Estimate': original_effects[key],
            'CI_Low': ci_low,
            'CI_High': ci_high,
            'p_value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })
    
    return pd.DataFrame(results)

In [ ]:
# Run significance testing for each dataset
print("Bootstrap Significance Testing (n=10,000 resamples)")
print("="*80)

significance_results = {}
for dataset in datasets + ['Average']:
    print(f"\n{dataset}:")
    print("-"*80)
    sig_df = compute_effect_significance(df.copy(), dataset)
    significance_results[dataset] = sig_df
    print(sig_df.to_string(index=False))

## 5. Visualizations

In [ ]:
# Main Effects Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
factors = ['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich']
factor_labels = ['Q1: Shortcuts', 'Q2: Expand Query', 'T1: OpenIE', 'T2: Enrich KB']

for idx, dataset in enumerate(datasets):
    ax = axes[idx // 2, idx % 2]
    
    x_positions = np.arange(len(factors))
    effects = [all_effects[dataset][f] for f in factors]
    colors = ['green' if e > 0 else 'red' for e in effects]
    
    bars = ax.bar(x_positions, effects, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(factor_labels, rotation=45, ha='right')
    ax.set_ylabel('Effect on Accuracy')
    ax.set_title(f'{dataset}: Main Effects')
    ax.set_ylim(-0.3, 0.3)
    
    # Add value labels
    for bar, effect in zip(bars, effects):
        height = bar.get_height()
        ax.annotate(f'{effect:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3 if height >= 0 else -12),
                    textcoords="offset points",
                    ha='center', va='bottom' if height >= 0 else 'top',
                    fontsize=9)

plt.tight_layout()
plt.savefig('main_effects_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pareto Chart of Effects (for average across datasets)
fig, ax = plt.subplots(figsize=(10, 6))

effect_names = list(avg_effects.keys())
effect_values = [abs(avg_effects[e]) for e in effect_names]
effect_signs = ['+' if avg_effects[e] > 0 else '-' for e in effect_names]

# Sort by absolute value
sorted_idx = np.argsort(effect_values)[::-1]
sorted_names = [effect_names[i] for i in sorted_idx]
sorted_values = [effect_values[i] for i in sorted_idx]
sorted_signs = [effect_signs[i] for i in sorted_idx]
sorted_colors = ['green' if s == '+' else 'red' for s in sorted_signs]

bars = ax.barh(range(len(sorted_names)), sorted_values, color=sorted_colors, alpha=0.7, edgecolor='black')
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels([f"{n} ({s})" for n, s in zip(sorted_names, sorted_signs)])
ax.set_xlabel('|Effect| on Accuracy')
ax.set_title('Pareto Chart: Effect Magnitudes (Average Across Datasets)')
ax.invert_yaxis()

# Add significance threshold line
ax.axvline(x=0.05, color='orange', linestyle='--', linewidth=2, label='Moderate threshold')
ax.axvline(x=0.10, color='red', linestyle='--', linewidth=2, label='Strong threshold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('pareto_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of effects across datasets
fig, ax = plt.subplots(figsize=(12, 6))

effect_matrix = pd.DataFrame(all_effects).T
effect_matrix = effect_matrix[['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich', 
                               'Q1*Q2 (=T1*T2)', 'Q1*T1 (=Q2*T2)', 'Q1*T2 (=Q2*T1)']]

sns.heatmap(effect_matrix, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-0.25, vmax=0.25, ax=ax, cbar_kws={'label': 'Effect on Accuracy'})
ax.set_title('Effect Heatmap Across Datasets')
ax.set_xlabel('Effect')
ax.set_ylabel('Dataset')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('effect_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interaction Plots for aliased pairs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Using average accuracy
response = 'Average'

# Q1 x Q2 interaction (aliased with T1 x T2)
ax = axes[0]
for q1_level in [-1, 1]:
    subset = df[df['Q1_shortcuts'] == q1_level]
    means = subset.groupby('Q2_expand')[response].mean()
    label = 'Q1=+' if q1_level == 1 else 'Q1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['Q2=-', 'Q2=+'])
ax.set_xlabel('Q2: Expand Query')
ax.set_ylabel('Average Accuracy')
ax.set_title('Q1 x Q2 Interaction\n(aliased with T1 x T2)')
ax.legend()

# Q1 x T1 interaction (aliased with Q2 x T2)
ax = axes[1]
for q1_level in [-1, 1]:
    subset = df[df['Q1_shortcuts'] == q1_level]
    means = subset.groupby('T1_openie')[response].mean()
    label = 'Q1=+' if q1_level == 1 else 'Q1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['T1=-', 'T1=+'])
ax.set_xlabel('T1: OpenIE')
ax.set_ylabel('Average Accuracy')
ax.set_title('Q1 x T1 Interaction\n(aliased with Q2 x T2)')
ax.legend()

# T1 x T2 interaction (aliased with Q1 x Q2)
ax = axes[2]
for t1_level in [-1, 1]:
    subset = df[df['T1_openie'] == t1_level]
    means = subset.groupby('T2_enrich')[response].mean()
    label = 'T1=+' if t1_level == 1 else 'T1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['T2=-', 'T2=+'])
ax.set_xlabel('T2: Enrich KB')
ax.set_ylabel('Average Accuracy')
ax.set_title('T1 x T2 Interaction\n(aliased with Q1 x Q2)')
ax.legend()

plt.tight_layout()
plt.savefig('interaction_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance by Configuration
fig, ax = plt.subplots(figsize=(12, 6))

# Create configuration labels
config_labels = []
for _, row in df.iterrows():
    label = f"Run {int(row['Run'])}\n"
    label += f"Q1:{'+' if row['Q1_shortcuts']==1 else '-'} "
    label += f"Q2:{'+' if row['Q2_expand']==1 else '-'}\n"
    label += f"T1:{'+' if row['T1_openie']==1 else '-'} "
    label += f"T2:{'+' if row['T2_enrich']==1 else '-'}"
    config_labels.append(label)

x = np.arange(len(config_labels))
width = 0.2

for i, dataset in enumerate(datasets):
    ax.bar(x + i*width - 1.5*width, df[dataset], width, label=dataset, alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_xlabel('Configuration')
ax.set_title('Accuracy by Configuration and Dataset')
ax.set_xticks(x)
ax.set_xticklabels(config_labels, fontsize=8)
ax.legend(loc='upper right')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('accuracy_by_config.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
# Best and worst configurations
print("Performance Summary")
print("="*80)

for dataset in datasets + ['Average']:
    best_idx = df[dataset].idxmax()
    worst_idx = df[dataset].idxmin()
    
    print(f"\n{dataset}:")
    print(f"  Mean: {df[dataset].mean():.3f}, Std: {df[dataset].std():.3f}")
    print(f"  Best:  Run {df.loc[best_idx, 'Run']:.0f} (Acc={df.loc[best_idx, dataset]:.3f})")
    print(f"         Q1={'+' if df.loc[best_idx, 'Q1_shortcuts']==1 else '-'}, "
          f"Q2={'+' if df.loc[best_idx, 'Q2_expand']==1 else '-'}, "
          f"T1={'+' if df.loc[best_idx, 'T1_openie']==1 else '-'}, "
          f"T2={'+' if df.loc[best_idx, 'T2_enrich']==1 else '-'}")
    print(f"  Worst: Run {df.loc[worst_idx, 'Run']:.0f} (Acc={df.loc[worst_idx, dataset]:.3f})")
    print(f"         Q1={'+' if df.loc[worst_idx, 'Q1_shortcuts']==1 else '-'}, "
          f"Q2={'+' if df.loc[worst_idx, 'Q2_expand']==1 else '-'}, "
          f"T1={'+' if df.loc[worst_idx, 'T1_openie']==1 else '-'}, "
          f"T2={'+' if df.loc[worst_idx, 'T2_enrich']==1 else '-'}")

In [ ]:
# Dataset difficulty ranking
print("\nDataset Difficulty (lower mean = harder):")
print("-"*40)
dataset_means = df[datasets].mean().sort_values()
for i, (ds, mean) in enumerate(dataset_means.items(), 1):
    print(f"  {i}. {ds}: {mean:.3f}")

## 7. Interpretation and Recommendations

In [ ]:
def generate_interpretation(effects, sig_results):
    """
    Generate natural language interpretation of results.
    """
    print("\n" + "="*80)
    print("INTERPRETATION SUMMARY")
    print("="*80)
    
    factors_info = {
        'Q1_shortcuts': ('Shortcut Detectors', 'use_shortcuts'),
        'Q2_expand': ('Query Expansion + Voting', 'expand_query'),
        'T1_openie': ('OpenIE Triples', 'use_openie'),
        'T2_enrich': ('KB Enrichment', 'use_enrichment_kb')
    }
    
    # Main effects interpretation
    print("\n1. MAIN EFFECTS (Average across datasets):")
    print("-"*60)
    
    sorted_factors = sorted(factors_info.keys(), key=lambda x: abs(effects[x]), reverse=True)
    
    for factor in sorted_factors:
        name, code = factors_info[factor]
        eff = effects[factor]
        
        # Get significance
        sig_row = sig_results[sig_results['Effect'] == factor]
        if not sig_row.empty:
            p_val = sig_row['p_value'].values[0]
            is_sig = sig_row['Significant'].values[0] == 'Yes'
        else:
            p_val = 1.0
            is_sig = False
        
        direction = "IMPROVES" if eff > 0 else "DECREASES"
        magnitude = abs(eff)
        
        if magnitude > 0.10:
            strength = "strongly"
        elif magnitude > 0.05:
            strength = "moderately"
        else:
            strength = "weakly"
        
        sig_marker = "*" if is_sig else ""
        
        print(f"  {name} ({code}): {direction} accuracy by {magnitude:.1%} {strength}{sig_marker}")
        if is_sig:
            print(f"      → Statistically significant (p={p_val:.3f})")
    
    # Interaction interpretation
    print("\n2. TWO-FACTOR INTERACTIONS (aliased pairs):")
    print("-"*60)
    
    interactions = [
        ('Q1*Q2 (=T1*T2)', 'Shortcuts × QueryExpand OR OpenIE × Enrichment'),
        ('Q1*T1 (=Q2*T2)', 'Shortcuts × OpenIE OR QueryExpand × Enrichment'),
        ('Q1*T2 (=Q2*T1)', 'Shortcuts × Enrichment OR QueryExpand × OpenIE')
    ]
    
    for key, desc in interactions:
        eff = effects[key]
        if abs(eff) > 0.05:
            print(f"  {key}: Effect = {eff:+.3f}")
            print(f"      → {desc}")
            if eff > 0:
                print("      → Synergistic: both factors together help more than expected")
            else:
                print("      → Antagonistic: factors interfere when combined")
    
    # Recommendations
    print("\n3. RECOMMENDATIONS:")
    print("-"*60)
    
    # Find best combination
    best_config = df.loc[df['Average'].idxmax()]
    print(f"  Best overall configuration:")
    for factor in sorted_factors:
        name, code = factors_info[factor]
        setting = 'ON' if best_config[factor] == 1 else 'OFF'
        print(f"    - {code}: {setting}")

# Run interpretation
generate_interpretation(avg_effects, significance_results['Average'])

## 8. Export Results for LaTeX

In [ ]:
def generate_latex_tables():
    """
    Generate LaTeX tables for the paper.
    """
    
    # Table 1: Raw results by configuration
    print("% LaTeX Table: Results by Configuration")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Accuracy (\\%) by configuration across datasets.}")
    print("\\label{tab:results}")
    print("\\begin{tabular}{c cccc | cccc | c}")
    print("\\toprule")
    print("Run & $Q_1$ & $Q_2$ & $T_1$ & $T_2$ & LogiQA2 & LogicBench & DocNLI & Alice & Avg \\\\")
    print("\\midrule")
    
    for _, row in df.iterrows():
        q1 = '+' if row['Q1_shortcuts'] == 1 else '-'
        q2 = '+' if row['Q2_expand'] == 1 else '-'
        t1 = '+' if row['T1_openie'] == 1 else '-'
        t2 = '+' if row['T2_enrich'] == 1 else '-'
        
        print(f"{int(row['Run'])} & ${q1}$ & ${q2}$ & ${t1}$ & ${t2}$ & "
              f"{row['LogiQA2']*100:.1f} & {row['LogicBench']*100:.1f} & "
              f"{row['DocNLI']*100:.1f} & {row['Alice']*100:.1f} & {row['Average']*100:.1f} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")
    print()
    
    # Table 2: Main effects
    print("% LaTeX Table: Main Effects")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Estimated main effects on accuracy. Positive values indicate the factor improves performance when enabled.}")
    print("\\label{tab:effects}")
    print("\\begin{tabular}{l cccc c}")
    print("\\toprule")
    print("Factor & LogiQA2 & LogicBench & DocNLI & Alice & Average \\\\")
    print("\\midrule")
    
    factors = ['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich']
    factor_names = ['$Q_1$: Shortcuts', '$Q_2$: Expand', '$T_1$: OpenIE', '$T_2$: Enrich']
    
    for factor, name in zip(factors, factor_names):
        vals = [all_effects[ds][factor] for ds in datasets]
        avg_val = avg_effects[factor]
        
        # Bold significant effects
        formatted = []
        for v in vals + [avg_val]:
            if abs(v) > 0.05:
                formatted.append(f"\\textbf{{{v*100:+.1f}}}")
            else:
                formatted.append(f"{v*100:+.1f}")
        
        print(f"{name} & {' & '.join(formatted)} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

generate_latex_tables()

In [ ]:
# Save results to CSV for external use
df.to_csv('results_with_design.csv', index=False)

# Save effects summary
effects_df = pd.DataFrame(all_effects)
effects_df['Average'] = pd.Series(avg_effects)
effects_df.to_csv('effects_summary.csv')

print("Results saved to:")
print("  - results_with_design.csv")
print("  - effects_summary.csv")
print("  - main_effects_plot.png")
print("  - pareto_effects.png")
print("  - effect_heatmap.png")
print("  - interaction_plots.png")
print("  - accuracy_by_config.png")

## 9. Comparison with Baselines (Optional)

If you have baseline results (e.g., standard RAG, Chain-of-Thought), enter them below.

In [ ]:
# ============================================================================
# OPTIONAL: Enter baseline results for comparison
# ============================================================================

baselines = {
    'RAG': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    },
    'CoT': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    },
    'GPT-4 Direct': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    }
}

# Convert to 0-1 scale if needed
for method in baselines:
    for ds in datasets:
        if baselines[method][ds] > 1:
            baselines[method][ds] /= 100.0

In [ ]:
# Compare best configuration to baselines
if any(baselines[m][datasets[0]] > 0 for m in baselines):
    print("Comparison with Baselines")
    print("="*80)
    
    # Best pipeline configuration
    best_row = df.loc[df['Average'].idxmax()]
    
    print(f"\n{'Method':<25} {'LogiQA2':>10} {'LogicBench':>12} {'DocNLI':>10} {'Alice':>10} {'Average':>10}")
    print("-"*80)
    
    # Print baselines
    for method, scores in baselines.items():
        avg = np.mean([scores[ds] for ds in datasets])
        print(f"{method:<25} {scores['LogiQA2']*100:>10.1f} {scores['LogicBench']*100:>12.1f} "
              f"{scores['DocNLI']*100:>10.1f} {scores['Alice']*100:>10.1f} {avg*100:>10.1f}")
    
    # Print best pipeline
    print("-"*80)
    print(f"{'Our Pipeline (best)':<25} {best_row['LogiQA2']*100:>10.1f} {best_row['LogicBench']*100:>12.1f} "
          f"{best_row['DocNLI']*100:>10.1f} {best_row['Alice']*100:>10.1f} {best_row['Average']*100:>10.1f}")
else:
    print("No baseline results entered. Skip this section or add baseline data above.")

## 10. Per-Category Analysis (Optional)

If you have per-category results (entailment, contradiction, uncertain, not_mentioned), enter them below for more detailed analysis.

In [ ]:
# ============================================================================
# OPTIONAL: Per-category accuracy breakdown
# ============================================================================
# Structure: {dataset: {run: {category: accuracy}}}

per_category_results = {
    # Example structure - fill in with your data
    # 'DocNLI': {
    #     1: {'entailment': 0.85, 'contradiction': 0.75, 'uncertain': 0.60, 'not_mentioned': 0.90},
    #     2: {'entailment': 0.87, 'contradiction': 0.78, 'uncertain': 0.62, 'not_mentioned': 0.92},
    #     # ... etc for all 8 runs
    # },
}

In [ ]:
if per_category_results:
    print("Per-Category Analysis")
    print("="*80)
    
    categories = ['entailment', 'contradiction', 'uncertain', 'not_mentioned']
    
    for dataset, run_data in per_category_results.items():
        print(f"\n{dataset}:")
        print("-"*60)
        
        # Build dataframe for this dataset
        cat_df = df.copy()
        for cat in categories:
            cat_df[cat] = [run_data[r][cat] for r in range(1, 9)]
        
        # Compute effects per category
        for cat in categories:
            effects = compute_effects(cat_df.copy(), cat)
            print(f"\n  {cat.upper()}:")
            for factor in ['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich']:
                print(f"    {factor}: {effects[factor]:+.3f}")
else:
    print("No per-category results entered.")

---

## Summary

This notebook provides:

1. **Design verification**: Confirms the $2^{4-1}_{IV}$ fractional factorial structure and aliasing
2. **Effect estimation**: Main effects and two-factor interactions for each dataset
3. **Statistical testing**: Bootstrap confidence intervals and significance tests
4. **Visualizations**: Main effects plots, Pareto charts, heatmaps, interaction plots
5. **LaTeX export**: Ready-to-use tables for your paper
6. **Interpretation**: Automated summary of findings and recommendations

**Next steps**:
1. Fill in your experimental results in Section 2
2. Run all cells
3. Copy the LaTeX tables from Section 8 into your paper
4. Use the generated plots in your paper
5. Adapt the interpretation text for your Results section